In [ ]:
import glob
import numpy as np
import pandas as pd

files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
path = [f for f in files if "fingerprint_dataset_normalized" in f][0]
df = pd.read_csv(path)

LABEL_COL = "label"

# Features = numeric columns except the label
X = df.drop(columns=[LABEL_COL]).select_dtypes(include="number")
feature_names = list(X.columns)
X = X.values.astype(float)

# Convert text labels (Healthy, Bearing, ...) into integers 0..K-1
classes, y = np.unique(df[LABEL_COL].values, return_inverse=True)
n_classes = len(classes)

print(X.shape, classes)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)

In [ ]:
def best_split(X, y, feat_idx, n_classes, min_leaf):
    """Search the given features for the threshold that gives the purest children.
    Returns (feature, threshold, weighted_gini) or None if no valid split exists."""
    n = len(y)
    best = None
    onehot = np.eye(n_classes)[y]                  # n x K matrix of class indicators

    for f in feat_idx:
        order = np.argsort(X[:, f])
        xs = X[order, f]
        left = np.cumsum(onehot[order], axis=0)    # class counts to the left of each cut
        total = left[-1]
        right = total - left

        nl = np.arange(1, n + 1)                   # left size for a cut after row i
        nr = n - nl

        # Gini impurity = 1 - sum(p^2); weighted by child size
        with np.errstate(divide="ignore", invalid="ignore"):
            gl = 1 - ((left / nl[:, None]) ** 2).sum(axis=1)
            gr = 1 - ((right / nr[:, None]) ** 2).sum(axis=1)
        score = nl * gl + nr * gr

        # A cut is valid only between two different values and if both sides are big enough
        valid = np.zeros(n, dtype=bool)
        valid[:-1] = xs[:-1] != xs[1:]
        valid &= (nl >= min_leaf) & (nr >= min_leaf)
        if not valid.any():
            continue

        score = np.where(valid, score, np.inf)
        i = np.argmin(score)
        if best is None or score[i] < best[2]:
            best = (f, (xs[i] + xs[i + 1]) / 2, score[i])
    return best

In [ ]:
class DecisionTree:
    def __init__(self, max_depth, n_sub_features, min_leaf, n_classes, n_features, rng):
        self.max_depth = max_depth
        self.min_leaf = min_leaf
        self.n_classes = n_classes
        self.rng = rng
        self.importance = np.zeros(n_features)
        # Per-level feature subsets: level d gets its own random set of features
        self.level_features = [
            rng.choice(n_features, size=n_sub_features, replace=False)
            for _ in range(max_depth)
        ]

    def fit(self, X, y):
        self.root = self._build(X, y, depth=0)
        return self

    def _build(self, X, y, depth):
        counts = np.bincount(y, minlength=self.n_classes)
        # Stop if: depth limit reached, node is pure, or too small to split
        if depth >= self.max_depth or (counts > 0).sum() == 1 or len(y) < 2 * self.min_leaf:
            return {"leaf": True, "pred": counts.argmax()}

        split = best_split(X, y, self.level_features[depth], self.n_classes, self.min_leaf)
        if split is None:
            return {"leaf": True, "pred": counts.argmax()}

        f, thr, child_score = split
        parent_gini = 1 - ((counts / len(y)) ** 2).sum()
        # Importance = how much impurity this split removed, weighted by node size
        self.importance[f] += len(y) * parent_gini - child_score

        mask = X[:, f] <= thr
        return {
            "leaf": False, "f": f, "thr": thr,
            "left": self._build(X[mask], y[mask], depth + 1),
            "right": self._build(X[~mask], y[~mask], depth + 1),
        }

    def predict(self, X):
        out = np.empty(len(X), dtype=int)
        self._predict(self.root, X, np.arange(len(X)), out)
        return out

    def _predict(self, node, X, idx, out):
        if node["leaf"]:
            out[idx] = node["pred"]
            return
        mask = X[idx, node["f"]] <= node["thr"]
        self._predict(node["left"], X, idx[mask], out)
        self._predict(node["right"], X, idx[~mask], out)

In [ ]:
class RandomForest:
    def __init__(self, n_trees=100, min_depth=6, max_depth=16,
                 n_sub_features=4, min_leaf=5, seed=42):
        self.n_trees = n_trees
        self.min_depth, self.max_depth = min_depth, max_depth
        self.n_sub_features = n_sub_features
        self.min_leaf = min_leaf
        self.rng = np.random.default_rng(seed)

    def fit(self, X, y, n_classes):
        self.n_classes = n_classes
        n, n_features = X.shape
        self.trees = []
        self.depths = []
        for t in range(self.n_trees):
            # 1) Bootstrap: random rows WITH replacement
            rows = self.rng.integers(0, n, size=n)
            # 2) Random depth for this tree
            depth = int(self.rng.integers(self.min_depth, self.max_depth + 1))
            # 3) Per-level feature subsets are drawn inside DecisionTree
            tree = DecisionTree(depth, self.n_sub_features, self.min_leaf,
                                n_classes, n_features, self.rng)
            tree.fit(X[rows], y[rows])
            self.trees.append(tree)
            self.depths.append(depth)
            if (t + 1) % 10 == 0:
                print(f"trained {t + 1}/{self.n_trees} trees")
        self.importance = sum(t.importance for t in self.trees)
        self.importance /= self.importance.sum()
        return self

    def predict(self, X):
        votes = np.zeros((len(X), self.n_classes))
        for tree in self.trees:
            votes[np.arange(len(X)), tree.predict(X)] += 1   # each tree casts one vote
        return votes.argmax(axis=1)                          # majority vote wins

In [ ]:
rf = RandomForest(n_trees=30, min_depth=6, max_depth=14, n_sub_features=4)
rf.fit(X_train, y_train, n_classes)

In [ ]:
import matplotlib.pyplot as plt

y_pred = rf.predict(X_test)

# Confusion matrix: rows = true class, columns = predicted class
cm = np.zeros((n_classes, n_classes), dtype=int)
for t, p in zip(y_test, y_pred):
    cm[t, p] += 1

accuracy = np.trace(cm) / cm.sum()
precision = np.diag(cm) / np.maximum(cm.sum(axis=0), 1)   # of predicted X, how many were X
recall    = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)   # of true X, how many we caught
f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)

print(f"Accuracy: {accuracy:.4f}\n")
print(pd.DataFrame({"precision": precision, "recall": recall, "f1": f1}, index=classes).round(4))

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(n_classes)); ax.set_xticklabels(classes, rotation=45)
ax.set_yticks(range(n_classes)); ax.set_yticklabels(classes)
for i in range(n_classes):
    for j in range(n_classes):
        ax.text(j, i, cm[i, j], ha="center", va="center")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.title("Custom Random Forest Confusion Matrix")
plt.tight_layout()
plt.savefig("/kaggle/working/rf_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
def accuracy_with_k_trees(rf, X, y, k):
    votes = np.zeros((len(X), rf.n_classes))
    for tree in rf.trees[:k]:
        votes[np.arange(len(X)), tree.predict(X)] += 1
    return (votes.argmax(axis=1) == y).mean()

for k in [1, 5, 10, 20, 30]:
    print(k, "trees:", round(accuracy_with_k_trees(rf, X_test, y_test, k), 4))

In [ ]:
files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
sdf   = pd.read_csv([f for f in files if "stress_dataset_normalized" in f][0])
smeta = pd.read_csv([f for f in files if "stress_metadata" in f][0])

Xs = sdf[feature_names].values.astype(float)      # same column order as training
ys = np.array([list(classes).index(l) for l in sdf["label"]])
ys_pred = rf.predict(Xs)

print("Stress-set accuracy:", round((ys_pred == ys).mean(), 4))
for c in range(n_classes):
    m = ys == c
    print(f"{classes[c]:18s} recall: {(ys_pred[m] == c).mean():.4f}")

In [ ]:
smeta["correct"] = ys_pred == ys
for lab, g in smeta[smeta.label != "Healthy"].groupby("label"):
    g = g.assign(bucket=pd.qcut(g["severity"], 4))
    print("\n" + lab)
    print(g.groupby("bucket", observed=True)["correct"].mean().round(3))

In [ ]:
meta = pd.read_csv([f for f in files if "fingerprint_metadata" in f][0])
idx = np.arange(len(X))
_, idx_test = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)  # same split as Cell 2
mt = meta.iloc[idx_test].reset_index(drop=True)
mt["correct"] = rf.predict(X_test) == y_test

for lab, g in mt[mt.label != "Healthy"].groupby("label"):
    g = g.assign(bucket=pd.qcut(g["severity"], 4))
    print("\n" + lab)
    print(g.groupby("bucket", observed=True)["correct"].mean().round(3))

In [ ]:
cm = np.zeros((n_classes, n_classes), dtype=int)
for t, p in zip(ys, ys_pred):
    cm[t, p] += 1
print(pd.DataFrame(cm, index=classes, columns=classes))